In [2]:
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os, plotly, tensorflow as tf, tensorflow.keras as keras # tensorflow_addons as tfa
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

In [3]:
vX = pd.read_csv('testX/testX.csv').set_index('id')
tYX = pd.read_csv('trainYX/trainYX.csv').set_index('id')

In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold,cross_validate,GroupKFold,cross_val_score,GridSearchCV,StratifiedGroupKFold,StratifiedShuffleSplit
from itertools import combinations, product
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,PolynomialFeatures, OneHotEncoder,  FunctionTransformer, Normalizer
from sklearn.pipeline import make_pipeline, Pipeline, clone
from sklearn.mixture import GaussianMixture
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA, LinearDiscriminantAnalysis as LDA
from sklearn import svm
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin


In [5]:
df_train = tYX
df_test = vX
X = df_train.drop(columns=['Y'])
y = df_train['Y']

In [6]:
class NucleotideCountProportion(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.new_features = ["A_ratio","T_ratio","G_ratio","C_ratio"]
    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            vals = X.iloc[:, 0].values
        elif isinstance(X, np.ndarray):
            vals = X.ravel()
        else:
            vals = np.asarray(X)

        counts = np.array([
            [s.count('A')/len(s), s.count('T')/len(s), s.count('G')/len(s), s.count('C')/len(s)]
            for s in vals], dtype=np.float32)

        return pd.DataFrame(counts,columns=self.new_features,index=X.index)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.new_features, dtype=object),

In [7]:
NCP = NucleotideCountProportion()
X_N_ratio = NCP.fit_transform(X)

In [9]:
from itertools import product
from collections import Counter

# Generate all possible pairs and triplets
pairs = ["".join(p) for p in product("ACGT", repeat=2)]
triplets = ["".join(p) for p in product("ACGT", repeat=3)]

def count_kmers(dna):
    dna = str(dna)[:200]

    # Count pairs
    pair_counts = Counter(dna[i:i+2] for i in range(len(dna) - 1))
    total_pairs = sum(pair_counts.get(p, 0) for p in pairs)

    pair_result = {
        p: pair_counts.get(p, 0) / total_pairs if total_pairs else 0
        for p in pairs
    }

    # Count triplets
    tri_counts = Counter(dna[i:i+3] for i in range(len(dna) - 2))
    total_tri = sum(tri_counts.get(t, 0) for t in triplets)

    tri_result = {
        t: tri_counts.get(t, 0) / total_tri if total_tri else 0
        for t in triplets
    }

    return {**pair_result, **tri_result}

# Create the complete dataset in one go
kmer_df = X["DNA"].apply(count_kmers).apply(pd.Series)

print(kmer_df.shape)

(100000, 80)


In [10]:
X_combined = pd.concat([X_N_ratio,kmer_df],axis=1)
X_combined['gc_proportion'] = X_combined['G_ratio'] + X_combined['C_ratio']
overlap_mask = (X_combined['gc_proportion'] >= 0.45) & (X_combined['gc_proportion'] <= 0.60)


In [11]:
X_baseline = X_combined[~overlap_mask]
y_baseline = y[~overlap_mask]

X_overlap = X_combined[overlap_mask]
y_overlap = y[overlap_mask]

In [12]:
def build_NN_model(input_dim):
    tf.keras.utils.set_random_seed(0)
    Init = keras.initializers.RandomNormal(seed=0)
    input_emb = keras.layers.Input(shape=(input_dim,),name="input_emb")
    input_nuc_norm = keras.layers.BatchNormalization(name="bn_1")(input_emb)
    h = keras.layers.Dense(64, use_bias=False, name="dense_0")(input_nuc_norm)
    h = keras.layers.Activation("relu", name="act_0")(h)
    h = keras.layers.Dense(32, use_bias=False, name="dense_11")(h)
    h = keras.layers.Activation("relu", name="act_11")(h)
    h = keras.layers.Dense(16, use_bias=False, name="dense_2")(h)
    h = keras.layers.Activation("relu", name="act_2")(h)
    out = keras.layers.Dense(1,activation="sigmoid", name='output')(h)
    m = keras.Model(inputs=input_emb, outputs=out)
    m.compile(loss="binary_crossentropy",                    #"sparse_categorical_crossentropy" "binary_crossentropy"
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),     #"sgd"
              # optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.2, nesterov=True),
              metrics=[
                        "accuracy",                      #'binary_accuracy'  "accuracy"
                      ])
    return m

In [13]:
def build_NN_model_wide(input_dim1,input_dim2):
    tf.keras.utils.set_random_seed(0)
    Init = keras.initializers.RandomNormal(seed=0)
    input_1 = keras.layers.Input(shape=(input_dim1,),name="input_1")
    input_2 = keras.layers.Input(shape=(input_dim2,),name="input_2")
    input_nuc_norm = keras.layers.BatchNormalization(name="bn_1")(input_1)
    h = keras.layers.Dense(64, use_bias=False, name="dense_0")(input_nuc_norm)
    h = keras.layers.Activation("relu", name="act_0")(h)
    h = keras.layers.Concatenate(name="late_fusion_concat")([h, input_2])
    h = keras.layers.Dense(32, use_bias=False, name="dense_11")(h)
    h = keras.layers.Activation("relu", name="act_11")(h)
    # h = keras.layers.Concatenate(name="late_fusion_concat2")([h, input_2])
    h = keras.layers.Dense(16, use_bias=False, name="dense_2")(h)
    h = keras.layers.Activation("relu", name="act_2")(h)
    out = keras.layers.Dense(1,activation="sigmoid", name='output')(h)
    m = keras.Model(inputs=[input_1,input_2], outputs=out)
    m.compile(loss="binary_crossentropy",                    #"sparse_categorical_crossentropy" "binary_crossentropy"
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),     #"sgd"
              # optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.2, nesterov=True),
              metrics=[
                        "accuracy",                      #'binary_accuracy'  "accuracy"
                      ])
    return m

In [14]:
def get_stratified_subset(X_data, y_data, fraction=0.25, seed=42):
    _, sub_idx = train_test_split(
        range(len(X_data)),
        test_size=fraction,
        stratify=y_data,
        random_state=seed
    )
    X_sub = X_data.iloc[sub_idx] if hasattr(X_data, 'iloc') else X_data[sub_idx]
    y_sub = y_data.iloc[sub_idx] if hasattr(y_data, 'iloc') else y_data[sub_idx]
    return X_sub, y_sub

In [16]:
selection = ['AA', 'AT', 'CG', 'TT', 'TA', 'CC']

### GMM Clustering on Subset of Overlap Portion
Evaluate unsupervised clustering performance of Gaussian Mixture Model (GMM) on a discriminative subset of dinucleotide pairs (`selection = ['AA', 'AT', 'CG', 'TT', 'TA', 'CC']`) on the overlap (difficult) portion:
- Evaluation on a 25% stratified sample subset of overlap
- Evaluation on the full overlap portion (clustering accuracy, ROC AUC, high-confidence margin separation, classification report)

### Combine GMM Features
Combine GMM cluster posterior probabilities with the best-performing NN model (`build_NN_model` on `pair_df_overlap + X_N_ratio_overlap + tri_df_overlap`) across the exact same 5-seed cross-validation splits (`seed = [2, 12, 22, 32, 42]`, 80/20 train/val split).

GMM is fitted strictly on the training fold features (`pair_df_overlap[selection]`) and evaluated on the validation fold to ensure leak-free evaluation.

In [ ]:
seed = [
        2,
        12,
        22,
        32,
        42
        ]
accu = []
f1 = []

# selection4 = [
#                 'AA', 'CG', 'TT', 'TA', 'GC',
#                 'G_ratio',
#                 'C_ratio',
#                 'gc_proportion',
#                 'CGC',
#                 'GCG',
#                 'TCG',
#                 'TTA'
#                 ]
for i, s in enumerate(seed):
    X_train, X_val, y_train, y_val = train_test_split(
        X_overlap,
        y_overlap,
        train_size=0.8,
        stratify=y_overlap,
        shuffle=True,
        random_state=s
    )
    
    # Fit GMM on training fold subset (leak-free)
    fold_scaler = StandardScaler()
    gmm_poly = PolynomialFeatures(degree=1, include_bias=False)
    X_train_poly = gmm_poly.fit_transform(X_train[selection])
    X_train_gmm_scaled = fold_scaler.fit_transform(X_train_poly)
    X_val_poly = gmm_poly.transform(X_val[selection])
    X_val_gmm_scaled = fold_scaler.transform(X_val_poly)
    
    gmm_fold = GaussianMixture(n_components=2, covariance_type='tied', random_state=42, max_iter=500)
    gmm_fold.fit(X_train_gmm_scaled)
    
    # Extract cluster posterior probability
    tr_gmm_prob = pd.DataFrame(
        gmm_fold.predict_proba(X_train_gmm_scaled)[:, 1:],
        index=X_train.index,
        columns=['gmm_prob']
    )
    va_gmm_prob = pd.DataFrame(
        gmm_fold.predict_proba(X_val_gmm_scaled)[:, 1:],
        index=X_val.index,
        columns=['gmm_prob']
    )
    
    # Combine GMM probabilities with base features
    X_train_comb = pd.concat([X_train, tr_gmm_prob], axis=1)
    X_val_comb = pd.concat([X_val, va_gmm_prob], axis=1)
    
    big_NN = build_NN_model(X_train_comb.shape[1])
    big_NN.fit(X_train_comb, y_train, epochs=10, batch_size=128, verbose=0)
    y_pred = big_NN.predict(X_val_comb, verbose=0).round(0).astype(int)
    
    acc = accuracy_score(y_val, y_pred)
    f1_val = f1_score(y_val, y_pred)
    accu += [acc]
    f1 += [f1_val]
    print(f"Fold {i} (seed {s:2d}): Accuracy = {acc:.6f}, F1 = {f1_val:.6f}")

print(f"\nMean CV Accuracy: {np.mean(accu)}")
print(f"Mean CV F1:       {np.mean(f1)}")

Fold 0 (seed  2): Accuracy = 0.894238, F1 = 0.867459
Fold 1 (seed 12): Accuracy = 0.899344, F1 = 0.871628
Fold 2 (seed 22): Accuracy = 0.889132, F1 = 0.858736
Fold 3 (seed 32): Accuracy = 0.892779, F1 = 0.866727
Fold 4 (seed 42): Accuracy = 0.892050, F1 = 0.864096

Mean CV Accuracy: 0.8935083880379284
Mean CV F1:       0.8657290882695607


0.8935083880379284 | GMM (selection) + NN CV score

In [343]:
seed = [
        2,
        12,
        22,
        32,
        42
        ]
accu = []
f1 = []

selection = ['AA', 'AT', 'CG', 'TT', 'TA', 'CC']

for i, s in enumerate(seed):
    X_train, X_val, y_train, y_val = train_test_split(
        X_overlap,
        y_overlap,
        train_size=0.8,
        stratify=y_overlap,
        shuffle=True,
        random_state=s
    )
    
    # Fit GMM on training fold subset (leak-free)
    fold_scaler = StandardScaler()
    gmm_poly = PolynomialFeatures(degree=1, include_bias=False)
    X_train_poly = gmm_poly.fit_transform(X_train[selection])
    X_train_gmm_scaled = fold_scaler.fit_transform(X_train_poly)
    X_val_poly = gmm_poly.transform(X_val[selection])
    X_val_gmm_scaled = fold_scaler.transform(X_val_poly)
    
    gmm_fold = GaussianMixture(n_components=2, covariance_type='tied', random_state=42, max_iter=500)
    gmm_fold.fit(X_train_gmm_scaled)
    
    # Extract cluster posterior probability
    tr_gmm_prob = pd.DataFrame(
        gmm_fold.predict_proba(X_train_gmm_scaled)[:, 1:],
        index=X_train.index,
        columns=['gmm_prob']
    )
    va_gmm_prob = pd.DataFrame(
        gmm_fold.predict_proba(X_val_gmm_scaled)[:, 1:],
        index=X_val.index,
        columns=['gmm_prob']
    )

    sel_col_trans = ColumnTransformer(
        transformers=[
            ("selection", PolynomialFeatures(degree=2), selection),
        ],remainder="passthrough",
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    nn_preprocessor_pipeline = Pipeline([
        ("sel_poly", nn_sel_col_trans),
    ]).set_output(transform="pandas")

    # Combine GMM probabilities with base features
    X_train_trans = nn_preprocessor_pipeline.fit_transform(X_train)
    X_val_trans = nn_preprocessor_pipeline.transform(X_val)
    
    big_NN = build_NN_model_wide(X_train.shape[1],tr_gmm_prob.shape[1])
    big_NN.fit([X_train,tr_gmm_prob], y_train, epochs=10, batch_size=128, verbose=0)
    y_pred = big_NN.predict([X_val,va_gmm_prob], verbose=0).round(0).astype(int)
    
    acc = accuracy_score(y_val, y_pred)
    f1_val = f1_score(y_val, y_pred)
    accu += [acc]
    f1 += [f1_val]
    print(f"Fold {i} (seed {s:2d}): Accuracy = {acc:.6f}, F1 = {f1_val:.6f}")

print(f"\nMean CV Accuracy: {np.mean(accu)}")
print(f"Mean CV F1:       {np.mean(f1)}")

Fold 0 (seed  2): Accuracy = 0.893144, F1 = 0.865658
Fold 1 (seed 12): Accuracy = 0.901167, F1 = 0.874711
Fold 2 (seed 22): Accuracy = 0.891320, F1 = 0.862165
Fold 3 (seed 32): Accuracy = 0.893508, F1 = 0.868587
Fold 4 (seed 42): Accuracy = 0.890956, F1 = 0.862781

Mean CV Accuracy: 0.8940189642596647
Mean CV F1:       0.8667803235629016


0.8940189642596647 | GMM (selection) + NN + poly features and late fusion model CV score

In [18]:
seed = [
        2,
        12,
        22,
        32,
        42
        ]
accu = []
f1 = []

for i, s in enumerate(seed):
    X_train, X_val, y_train, y_val = train_test_split(
        X_baseline,
        y_baseline,
        train_size=0.85,
        stratify=y_baseline,
        # shuffle=True,
        random_state=s
    )
    model = build_NN_model(X_train.shape[1])
    # model = svm.SVC(kernel='rbf',C=1.3,gamma=0.01)
    # model = LDA()
    
    model.fit(X_train,y_train,
                epochs=10,batch_size=128,verbose=0
                )
    y_pred = model.predict(X_val).round(0).astype(int)
    accu+=[accuracy_score(y_val,y_pred)]
print(np.mean(accu),np.std(accu))

405/405 ━━━━━━━━━━━━━━━━━━━━ 0s 375us/step
405/405 ━━━━━━━━━━━━━━━━━━━━ 0s 382us/step
405/405 ━━━━━━━━━━━━━━━━━━━━ 0s 366us/step
405/405 ━━━━━━━━━━━━━━━━━━━━ 0s 363us/step
405/405 ━━━━━━━━━━━━━━━━━━━━ 0s 357us/step
0.9980222496909764 0.00017343900124105295


In [17]:
seed = [
        2,
        12,
        22,
        32,
        42
        ]
accu = []
f1 = []

for i, s in enumerate(seed):
    X_train, X_val, y_train, y_val = train_test_split(
        X_baseline,
        y_baseline,
        train_size=0.9,
        stratify=y_baseline,
        random_state=s
    )

    X_train2, X_val2, y_train2, y_val2 = train_test_split(
        X_train,
        y_train,
        train_size=0.8,
        stratify=y_train,
        random_state=s+17
    )

    X_lda, y_lda = get_stratified_subset(X_train2, y_train2, fraction=0.95, seed=s+1)
    X_nn1, y_nn1   = get_stratified_subset(X_train2, y_train2, fraction=0.95, seed=s+3)

    lda_sel_col_trans = ColumnTransformer(
        transformers=[
            ("selection", PolynomialFeatures(degree=2), selection),
        ],remainder="passthrough",
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    lda_pipeline = Pipeline([
        ("sel_poly", lda_sel_col_trans),
        ('lda',LDA()),
    ])

    nn_sel_col_trans = ColumnTransformer(
        transformers=[
            ("selection", PolynomialFeatures(degree=2), selection),
        ],remainder="passthrough",
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    nn_preprocessor_pipeline = Pipeline([
        ("sel_poly", nn_sel_col_trans),
    ]).set_output(transform="pandas")

    X_nn1_trans = nn_preprocessor_pipeline.fit_transform(X_nn1)
    X_val2_trans = nn_preprocessor_pipeline.transform(X_val2)
    X_val_trans = nn_preprocessor_pipeline.transform(X_val)

    #------------model 1--------------
    lda_pipeline.fit(X_lda,y_lda)
    y_prob_lda = lda_pipeline.predict_proba(X_val2)[:,1]

    #------------model 2--------------
    NN1_m = build_NN_model(X_nn1_trans.shape[1])
    NN1_m.fit(
        X_nn1_trans,
        y_nn1,
        epochs=10,
        batch_size=128,
        verbose=0
    )
    y_prob_nn1 = NN1_m.predict(X_val2_trans,verbose=0).reshape(-1)

    prob_df = pd.DataFrame(np.column_stack((
                                            X_val2_trans.values,
                                            y_prob_lda,
                                            y_prob_nn1,
                                            ))
                            ,index=X_val2.index)

    # meta_ss = StandardScaler()
    # meta_clf = LDA()
    meta_clf = build_NN_model(prob_df.shape[1])
    meta_clf.fit(   prob_df,y_val2,
                    epochs=15,
                    batch_size=128,
                    verbose=0
            )
    # meta_clf.fit(prob_df,y_val2)
    # meta_clf.fit(meta_ss.fit_transform(prob_df),y_val2)

    y_prob_lda_val = lda_pipeline.predict_proba(X_val)[:,1]
    y_prob_nn1_val = NN1_m.predict(X_val_trans,verbose=0).reshape(-1)

    print(accuracy_score(y_val,y_prob_lda_val.round(0).astype(int)))
    print(accuracy_score(y_val,y_prob_nn1_val.round(0).astype(int)))

    prob_df_final = pd.DataFrame(np.column_stack((
                                            X_val_trans.values,
                                            y_prob_lda_val,
                                            y_prob_nn1_val,
                                            )),
                            index=X_val.index)

    y_pred = meta_clf.predict(prob_df_final).round(0).astype(int)
    # y_pred = meta_clf.predict(meta_ss.transform(prob_df_final)).round(0).astype(int)

    a = accuracy_score(y_val, y_pred)
    f = f1_score(y_val, y_pred)
    accu += [a]
    f1 += [f]
    print(f'Accuracy: {a:.4f}, F1: {f:.4f}')

print("mean | std | f1 | std")
print(f'{np.mean(accu):.4f} | {np.std(accu):.4f} | ', end="")
print(f'{np.mean(f1):.4f} | {np.std(f1):.4f}')


0.9984936268829664
0.9981460023174971
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 443us/step
Accuracy: 0.9983, F1: 0.9983
0.9976825028968713
0.9980301274623407
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 395us/step
Accuracy: 0.9978, F1: 0.9979
0.9986095017381228
0.9984936268829664
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 407us/step
Accuracy: 0.9986, F1: 0.9986
0.9977983777520278
0.9977983777520278
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 434us/step
Accuracy: 0.9981, F1: 0.9982
0.9980301274623407
0.9980301274623407
270/270 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step
Accuracy: 0.9977, F1: 0.9977
mean | std | f1 | std
0.9981 | 0.0003 | 0.9981 | 0.0003
